In [ ]:
%%bash
set -eo pipefail

root="$PWD"
work="$root/archives/cryo-benchmark"
if [ "$(uname -sm)" != "Linux x86_64" ]; then
    echo "EModelX requires x86-64 Linux and CUDA." >&2
    exit 1
fi
mkdir -p "$work"

if ! dpkg -s g++ libgl1 >/dev/null 2>&1; then
    apt-get update
    apt-get install -y g++ libgl1
fi

conda="$(command -v conda || printf '%s' "$work/miniforge/bin/conda")"
if [ ! -x "$conda" ]; then
    curl -fLo "$work/miniforge.sh" https://github.com/conda-forge/miniforge/releases/latest/download/Miniforge3-Linux-x86_64.sh
    bash "$work/miniforge.sh" -b -p "$work/miniforge"
fi

repo="$work/EModelX"
if [ ! -d "$repo" ]; then
    git clone https://github.com/biomed-AI/EModelX.git "$repo"
fi
git -C "$repo" checkout --quiet --detach 53713cf

env="$work/EModelX-env"
CONDA_CHANNEL_PRIORITY=flexible "$conda" env update -p "$env" -f "$repo/EModelX.yml"

"$conda" run -p "$env" python -m pip install gdown

cd "$repo"
mkdir -p models
if [ ! -s models/best_CA_model.ckpt ]; then
    "$env/bin/gdown" 1n8yiVr4oRxsY4ii2a05hcx5gI3eIybA1 -O models/best_CA_model.ckpt
fi
if [ ! -s models/best_BB_model.ckpt ]; then
    "$env/bin/gdown" 1GbmezR4tVCYaALZ330TL4vk84kGDQD3V -O models/best_BB_model.ckpt
fi
if [ ! -s models/best_AA_model.ckpt ]; then
    "$env/bin/gdown" 1evghUiUcb3mFiCQ2-Gjez9AtA2w6Pyir -O models/best_AA_model.ckpt
fi


In [ ]:
%%bash
set -eo pipefail

root="$PWD"
work="$root/archives/cryo-benchmark"

conda="$(command -v conda || printf '%s' "$work/miniforge/bin/conda")"

method=EModelX
export MPLBACKEND=Agg
cd "$work/EModelX"

for target in 22XP 32MC; do
    data="$root/data/targets/$target"
    map=("$data"/inputs/emd_*.map)
    fasta="$data/inputs/rcsb_pdb_${target}.fasta"
    out="$work/runs/$method/$target"

    templates="$out/templates"
    mkdir -p "$templates/${target}_1"
    cp "$data"/predictions/af2/*/*_unrelaxed_rank_001_*.pdb "$templates/${target}_1/ranked_0.pdb"
    time "$conda" run -p "$work/EModelX-env" --no-capture-output python -u run.py \
        --protocol=temp_flex --EM_map "${map[0]}" --fasta "$fasta" --template_dir "$templates" \
        --output_dir "$out" --run_pulchra --pulchra_path modules/pulchra304/src/pulchra --mul_proc_num 4
done


In [ ]:
%%bash
set -eo pipefail

root="$PWD"
work="$root/archives/cryo-benchmark"

if [ "$(uname -s)" = Linux ] && ! command -v g++ >/dev/null; then
    apt-get update
    apt-get install -y g++
fi

phenix_env="${PHENIX_ENV:-$work/phenix/phenix_env.sh}"
if [ ! -f "$phenix_env" ]; then
    for script in "$root/../cryo-benchmark-work/phenix/phenix_env.sh" "$HOME"/phenix-*/phenix_env.sh /Applications/phenix-*/phenix_env.sh; do
        if [ -f "$script" ]; then phenix_env="$script"; break; fi
    done
fi

method="EModelX"
if [ ! -d "$work/USalign" ]; then
    git clone https://github.com/pylelab/USalign.git "$work/USalign"
fi
make -C "$work/USalign" USalign

for target in 22XP 32MC; do
    out="$work/runs/$method/$target"
    reference="$root/data/targets/$target/reference/$target.cif"
    model="$out/rcsb_pdb_${target}_EModelX(+AF)_all_atom_model.pdb"
    test -s "$model"
    "$work/USalign/USalign" "$model" "$reference" -mol prot -mm 1 -ter 1 -outfmt 2 > "$out/usalign.tsv"
    if [ -f "$phenix_env" ]; then
        (source "$phenix_env"; phenix.chain_comparison "$reference" "$model" \
            max_dist=3 unique_part_of_target_only=False test_unique_part_of_target_only=False \
            use_crystal_symmetry=False) > "$out/chain-comparison.txt"
    fi
done
